# GPT-2 Core Model Training - Google Colab Version

This notebook trains a GPT-2 model on two datasets:
1. **DailyDialog**: Conversational dialog with emotions and acts
2. **Surgical Robotics**: Robot control instruction-response pairs

**Setup Required**: 

1. **Google Colab with GPU runtime enabled**
2. **Automatic file upload** (no Google Drive mounting needed)
3. **Upload your dataset files** using the instructions below

## Install Dependencies

In [2]:
# Check for GPU availability
import torch
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


## Mount Google Drive

In [15]:
# Mount Google Drive with error handling
from google.colab import drive
import os

print("Attempting to mount Google Drive...")
print("IMPORTANT: You'll need to authorize access in the popup window")
print("="*60)

try:
    # Force remount if there's an existing mount
    if os.path.exists('/content/drive'):
        print("⚠️  Existing mount detected, forcing remount...")
        drive.mount('/content/drive', force_remount=True)
    else:
        drive.mount('/content/drive')
    
    print("\n✓ Google Drive mounted successfully!")
    print(f"✓ Mount point exists: {os.path.exists('/content/drive/My Drive')}")
    
except Exception as e:
    print(f"\n❌ Mount failed: {e}")
    print("\nTROUBLESHOOTING STEPS:")
    print("="*60)
    print("1. Click 'Connect to Google Drive' in the popup")
    print("2. Select your Google account")
    print("3. Click 'Allow' to grant Colab access to Drive")
    print("\nIf the popup doesn't appear:")
    print("• Check if popups are blocked in your browser")
    print("• Try refreshing the page and running again")
    print("• Clear browser cache/cookies")
    print("\nIf you still have issues:")
    print("• Restart runtime: Runtime → Restart Runtime")
    print("• Try in an incognito/private browser window")
    print("• Check Google Drive status: https://www.google.com/appsstatus")
    print("="*60)
    raise

Attempting to mount Google Drive...
IMPORTANT: You'll need to authorize access in the popup window

❌ Mount failed: mount failed

TROUBLESHOOTING STEPS:
1. Click 'Connect to Google Drive' in the popup
2. Select your Google account
3. Click 'Allow' to grant Colab access to Drive

If the popup doesn't appear:
• Check if popups are blocked in your browser
• Try refreshing the page and running again
• Clear browser cache/cookies

If you still have issues:
• Restart runtime: Runtime → Restart Runtime
• Try in an incognito/private browser window
• Check Google Drive status: https://www.google.com/appsstatus

❌ Mount failed: mount failed

TROUBLESHOOTING STEPS:
1. Click 'Connect to Google Drive' in the popup
2. Select your Google account
3. Click 'Allow' to grant Colab access to Drive

If the popup doesn't appear:
• Check if popups are blocked in your browser
• Try refreshing the page and running again
• Clear browser cache/cookies

If you still have issues:
• Restart runtime: Runtime → Resta

ValueError: mount failed

In [ ]:
# Verify Drive Structure and Locate Dataset
import os

print("VERIFYING GOOGLE DRIVE STRUCTURE")
print("="*50)

# Check if Drive is mounted
if os.path.exists('/content/drive'):
    print("✓ Drive is mounted\n")
    
    # List My Drive contents
    print("=== Contents of My Drive ===")
    mydrive_path = '/content/drive/My Drive'
    if os.path.exists(mydrive_path):
        items = os.listdir(mydrive_path)
        for item in items:
            print(f"  - {item}")
    
    # Check for SAR-DL-Podcast folder
    print("\n=== Looking for SAR-DL-Podcast ===")
    sar_path = '/content/drive/My Drive/SAR-DL-Podcast'
    if os.path.exists(sar_path):
        print(f"✓ Found: {sar_path}")
        print("\nContents:")
        for item in os.listdir(sar_path):
            print(f"  - {item}")
        
        # Check for dataset folder
        dataset_path = os.path.join(sar_path, 'dataset')
        if os.path.exists(dataset_path):
            print(f"\n✓ Found dataset folder")
            print("\nDataset contents:")
            for item in os.listdir(dataset_path):
                item_path = os.path.join(dataset_path, item)
                if os.path.isdir(item_path):
                    print(f"  📁 {item}/")
                else:
                    print(f"  📄 {item}")
            
            # Check DailyDialog
            daily_dialog_path = os.path.join(dataset_path, 'DailyDialog')
            if os.path.exists(daily_dialog_path):
                print(f"\n=== DailyDialog folder contents ===")
                for item in os.listdir(daily_dialog_path):
                    item_path = os.path.join(daily_dialog_path, item)
                    if os.path.isdir(item_path):
                        print(f"  📁 {item}/")
                        # List contents of subdirectories
                        for subitem in os.listdir(item_path):
                            print(f"      - {subitem}")
                    else:
                        print(f"  📄 {item}")
            
            # Check Surgical_Robotics
            robot_path = os.path.join(dataset_path, 'Surgical_Robotics')
            if os.path.exists(robot_path):
                print(f"\n=== Surgical_Robotics folder contents ===")
                for item in os.listdir(robot_path):
                    print(f"  📄 {item}")
        else:
            print(f"❌ dataset folder not found at: {dataset_path}")
    else:
        print(f"❌ SAR-DL-Podcast not found at: {sar_path}")
        print("\nPlease create the folder structure in Google Drive:")
        print("  My Drive/SAR-DL-Podcast/dataset/")
else:
    print("❌ Drive is not mounted!")
    print("Please run the previous cell to mount Google Drive.")

## Configuration and Data Verification

In [11]:
import os
import yaml
import sys
import json
import numpy as np
import matplotlib.pyplot as plt

print("CONFIGURATION SETUP")
print("="*50)

# ========================================
# CONFIGURATION
# ========================================

# Hardcoded configuration
config = {
    'learning_rate': 5e-5,
    'batch_size': 8,
    'num_epochs': 3,
    'max_length': 512,
    'gradient_accumulation_steps': 4,
    'robot_train_split': 0.7,
    'generation_max_length': 150,
    'generation_temperature': 0.7,
    'generation_top_p': 0.9,
    'eval_sample_size': 100,
    'output_dir': '/content/core_results',
    'best_model_path': '/content/core_results/gpt2_best_model',
    'final_model_path': '/content/core_results/gpt2_final',
    'plot_path': '/content/core_results/training_curves.png'
}

print("✓ Configuration loaded")
print(f"  Learning Rate: {config['learning_rate']}")
print(f"  Batch Size: {config['batch_size']}")
print(f"  Epochs: {config['num_epochs']}")

# Set data paths for Google Drive
dataset_base = '/content/drive/My Drive/SAR-DL-Podcast/dataset'

LOCAL_DAILY_DIALOG_DIR = os.path.join(dataset_base, 'DailyDialog')
LOCAL_ROBOT_CONTROL_PATH = os.path.join(dataset_base, 'Surgical_Robotics', 'robot_control.json')

config['robot_control_path'] = LOCAL_ROBOT_CONTROL_PATH

print(f"\n✓ Dataset paths set:")
print(f"  DailyDialog: {LOCAL_DAILY_DIALOG_DIR}")
print(f"  Robot Control: {LOCAL_ROBOT_CONTROL_PATH}")


CONFIGURATION SETUP
✓ Configuration loaded
  Learning Rate: 5e-05
  Batch Size: 8
  Epochs: 3

✓ Dataset paths set:
  DailyDialog: /content/drive/My Drive/SAR-DL-Podcast/dataset/DailyDialog
  Robot Control: /content/drive/My Drive/SAR-DL-Podcast/dataset/Surgical_Robotics/robot_control.json


## Load GPT-2 Model and Tokenizer

In [3]:
# Load GPT-2 model and tokenizer directly
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Loading GPT-2 model and tokenizer...")
model_name = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set pad token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"✓ Model loaded successfully!")
print(f"Device: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading GPT-2 model and tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✓ Model loaded successfully!
Device: cuda
Model parameters: 124,439,808


## Daily Dialog Data Preparation

In [12]:
# Load dialog datasets from downloaded files
print("Loading DailyDialog dataset from downloaded files...")
daily_dialog_path = LOCAL_DAILY_DIALOG_DIR

def load_daily_dialog(data_dir):
    """Load DailyDialog data from text files"""
    utterances = []
    acts = []
    emotions = []
    
    # Determine which files to load based on directory
    if 'train' in data_dir:
        dialog_file = 'dialogues_train.txt'
        act_file = 'dialogues_act_train.txt'
        emotion_file = 'dialogues_emotion_train.txt'
    else:
        dialog_file = 'dialogues_validation.txt'
        act_file = 'dialogues_act_validation.txt'
        emotion_file = 'dialogues_emotion_validation.txt'
    
    with open(os.path.join(data_dir, dialog_file), 'r', encoding='utf-8') as f:
        for line in f:
            utterances.append(line.strip().split('__eou__')[:-1])
    
    with open(os.path.join(data_dir, act_file), 'r', encoding='utf-8') as f:
        for line in f:
            acts.append([int(a) for a in line.strip().split()])
    
    with open(os.path.join(data_dir, emotion_file), 'r', encoding='utf-8') as f:
        for line in f:
            emotions.append([int(e) for e in line.strip().split()])
    
    return utterances, acts, emotions

# Load training and validation data
train_utterances, train_acts, train_emotions = load_daily_dialog(os.path.join(daily_dialog_path, 'train'))
val_utterances, val_acts, val_emotions = load_daily_dialog(os.path.join(daily_dialog_path, 'validation'))

print(f"✓ Loaded {len(train_utterances)} training dialogs and {len(val_utterances)} validation dialogs")

Loading DailyDialog dataset from downloaded files...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/My Drive/SAR-DL-Podcast/dataset/DailyDialog/train/dialogues_train.txt'

In [ ]:
# Flatten dialogs into instruction-response pairs with context
dialog_train_instructions = []
dialog_train_responses = []

for idx, utterances in enumerate(train_utterances):
    acts = train_acts[idx]
    emotions = train_emotions[idx]
    
    for i in range(len(utterances) - 1):
        context_instruction = f"[Act: {acts[i]}] [Emotion: {emotions[i]}] {utterances[i]}"
        context_response = f"[Act: {acts[i+1]}] [Emotion: {emotions[i+1]}] {utterances[i + 1]}"
        dialog_train_instructions.append(context_instruction)
        dialog_train_responses.append(context_response)

# Same for validation
dialog_val_instructions = []
dialog_val_responses = []

for idx, utterances in enumerate(val_utterances):
    acts = val_acts[idx]
    emotions = val_emotions[idx]
    
    for i in range(len(utterances) - 1):
        context_instruction = f"[Act: {acts[i]}] [Emotion: {emotions[i]}] {utterances[i]}"
        context_response = f"[Act: {acts[i+1]}] [Emotion: {emotions[i+1]}] {utterances[i + 1]}"
        dialog_val_instructions.append(context_instruction)
        dialog_val_responses.append(context_response)

print(f"Created {len(dialog_train_instructions)} training pairs and {len(dialog_val_instructions)} validation pairs")

In [ ]:
# Tokenize daily dialog data - combine instruction and response for causal LM
dialog_train_combined = [inst + " " + resp for inst, resp in zip(dialog_train_instructions, dialog_train_responses)]
dialog_val_combined = [inst + " " + resp for inst, resp in zip(dialog_val_instructions, dialog_val_responses)]

dialog_train_encodings = tokenizer(dialog_train_combined, padding=True, truncation=True, return_tensors='pt', max_length=config['max_length'])
dialog_val_encodings = tokenizer(dialog_val_combined, padding=True, truncation=True, return_tensors='pt', max_length=config['max_length'])

# For causal LM, input_ids and labels are the same
dialog_train_input_ids = dialog_train_encodings['input_ids']
dialog_train_attention_mask = dialog_train_encodings['attention_mask']
dialog_train_labels = dialog_train_encodings['input_ids'].clone()

dialog_val_input_ids = dialog_val_encodings['input_ids']
dialog_val_attention_mask = dialog_val_encodings['attention_mask']
dialog_val_labels = dialog_val_encodings['input_ids'].clone()

print(f"Dialog Data Shapes:")
print(f"  Input IDs: {dialog_train_input_ids.shape}")
print(f"  Labels: {dialog_train_labels.shape}")

## Surgical Robotics Data Preparation

In [ ]:
# Load robot control data from downloaded file
print("Loading robot control data...")
with open(config['robot_control_path'], 'r') as f:
    robot_control_data = json.load(f)

instructions = [item['instruction'] for item in robot_control_data]
responses = [item['response'] for item in robot_control_data]

# Combine instruction and response for causal LM
robot_combined = [inst + " " + resp for inst, resp in zip(instructions, responses)]
robot_encodings = tokenizer(robot_combined, padding=True, truncation=True, return_tensors='pt', max_length=config['max_length'])

input_ids = robot_encodings['input_ids']
attention_mask = robot_encodings['attention_mask']
labels = robot_encodings['input_ids'].clone()

# Split into train/val
total_samples = len(robot_control_data)
train_size = int(config['robot_train_split'] * total_samples)

train_instruction_ids = input_ids[:train_size]
train_attention_mask = attention_mask[:train_size]
train_response = labels[:train_size]

val_instructions_ids = input_ids[train_size:]
val_attention_mask = attention_mask[train_size:]
val_response = labels[train_size:]

print(f"✓ Robot Control Data Loaded:")
print(f"  Total samples: {total_samples}")
print(f"  Training: {train_size}, Validation: {total_samples - train_size}")

## Combine and Balance Datasets

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, ConcatDataset
from torch.nn.utils.rnn import pad_sequence

# Custom collate function
def collate_fn(batch):
    input_ids = [item[0] for item in batch]
    attention_masks = [item[1] for item in batch]
    labels = [item[2] for item in batch]
    
    input_ids_padded = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_masks_padded = pad_sequence(attention_masks, batch_first=True, padding_value=0)
    labels_padded = pad_sequence(labels, batch_first=True, padding_value=tokenizer.pad_token_id)
    
    return input_ids_padded, attention_masks_padded, labels_padded

# Create datasets
dialog_train_dataset = TensorDataset(dialog_train_input_ids, dialog_train_attention_mask, dialog_train_labels)
dialog_val_dataset = TensorDataset(dialog_val_input_ids, dialog_val_attention_mask, dialog_val_labels)
robot_train_dataset = TensorDataset(train_instruction_ids, train_attention_mask, train_response)
robot_val_dataset = TensorDataset(val_instructions_ids, val_attention_mask, val_response)

# Balance datasets through oversampling
repeat_factor = max(1, len(dialog_train_dataset) // len(robot_train_dataset))
robot_train_repeated = ConcatDataset([robot_train_dataset] * repeat_factor)

print(f"Dataset Balancing:")
print(f"  Dialog samples: {len(dialog_train_dataset)}")
print(f"  Robot samples (original): {len(robot_train_dataset)}")
print(f"  Robot samples (oversampled {repeat_factor}x): {len(robot_train_repeated)}")

# Combine datasets
combined_train_dataset = ConcatDataset([dialog_train_dataset, robot_train_repeated])
combined_val_dataset = ConcatDataset([dialog_val_dataset, robot_val_dataset])

# Create dataloaders
train_loader = DataLoader(combined_train_dataset, batch_size=config['batch_size'], shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(combined_val_dataset, batch_size=config['batch_size'], collate_fn=collate_fn)

print(f"\nTotal Training Samples: {len(combined_train_dataset)}")
print(f"Training Batches: {len(train_loader)}")

## Training Setup

In [ ]:
from torch.optim import AdamW
from tqdm import tqdm

optimizer = AdamW(model.parameters(), lr=config['learning_rate'])
model.to(device)
model.train()

print(f"Training Configuration:")
print(f"  Device: {device}")
print(f"  Learning Rate: {config['learning_rate']}")
print(f"  Epochs: {config['num_epochs']}")
print(f"  Gradient Accumulation: {config['gradient_accumulation_steps']}")

## Training Loop

In [ ]:
best_val_loss = float('inf')
train_losses = []
val_losses = []

print("="*50)
print("Starting Training...")
print("="*50)

for epoch in range(config['num_epochs']):
    # Training
    model.train()
    total_train_loss = 0
    train_steps = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['num_epochs']}")
    
    for batch_idx, batch in enumerate(progress_bar):
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss / config['gradient_accumulation_steps']
        loss.backward()
        
        if (batch_idx + 1) % config['gradient_accumulation_steps'] == 0:
            optimizer.step()
            optimizer.zero_grad()
        
        total_train_loss += loss.item() * config['gradient_accumulation_steps']
        train_steps += 1
        progress_bar.set_postfix({'loss': f"{loss.item() * config['gradient_accumulation_steps']:.4f}"})
    
    avg_train_loss = total_train_loss / train_steps
    
    # Validation
    model.eval()
    total_val_loss = 0
    val_steps = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            input_ids = batch[0].to(device)
            attention_mask = batch[1].to(device)
            labels = batch[2].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_val_loss += outputs.loss.item()
            val_steps += 1
    
    avg_val_loss = total_val_loss / val_steps
    
    # Calculate perplexity
    train_perplexity = np.exp(avg_train_loss)
    val_perplexity = np.exp(avg_val_loss)
    
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    
    print(f"\nEpoch {epoch+1} Results:")
    print(f"  Training Loss: {avg_train_loss:.4f} | Perplexity: {train_perplexity:.2f}")
    print(f"  Validation Loss: {avg_val_loss:.4f} | Perplexity: {val_perplexity:.2f}")
    
    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        print(f"  ✓ New best model! Saving...")
        model.save_pretrained(config['best_model_path'])
        tokenizer.save_pretrained(config['best_model_path'])
    print("-" * 50)

# Save final model
model.save_pretrained(config['final_model_path'])
tokenizer.save_pretrained(config['final_model_path'])

print(f"\nTraining Complete!")
print(f"Best Validation Loss: {best_val_loss:.4f}")

## Plot Training Curves

In [ ]:
plt.figure(figsize=(10, 6))
epochs_range = range(1, config['num_epochs'] + 1)

plt.plot(epochs_range, train_losses, 'b-o', label='Training Loss', linewidth=2, markersize=8)
plt.plot(epochs_range, val_losses, 'r-s', label='Validation Loss', linewidth=2, markersize=8)

plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training and Validation Loss Over Epochs', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(config['plot_path'], dpi=300, bbox_inches='tight')
print(f"Training curves saved to: {config['plot_path']}")
plt.show()

## Evaluation Functions

In [ ]:
def calculate_distinct_n(texts, n):
    """Calculate distinct-n metric for diversity"""
    all_ngrams = []
    for text in texts:
        tokens = text.split()
        ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
        all_ngrams.extend(ngrams)
    
    if len(all_ngrams) == 0:
        return 0.0
    return len(set(all_ngrams)) / len(all_ngrams)

## Test Dialog Responses

In [ ]:
model.eval()

dialog_test_prompts = [
    "[Act: 1] [Emotion: 0] Hello, how are you today?",
    "[Act: 2] [Emotion: 3] I'm feeling really excited about this project!",
    "[Act: 4] [Emotion: 1] Could you help me with this problem?"
]

dialog_responses = []

print("Dialog Responses:")
print("="*50)

for i, prompt in enumerate(dialog_test_prompts):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=config['generation_max_length'],
            temperature=config['generation_temperature'],
            top_p=config['generation_top_p'],
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    dialog_responses.append(response)
    print(f"\n{i+1}. Prompt: {prompt}")
    print(f"   Response: {response}")
    print("-" * 50)

## Test Surgical Robotics Responses

In [ ]:
robotics_test_prompts = [
    "The vision system detects 'Preparation'. What robotic algorithm applies here?",
    "Explain the control theory behind robotic Dissection.",
    "The vision system detects the tool 'Grasper'. What is the robotic equivalent?",
    "Why is the robotic approach to 'Clipping/Cutting' considered safer?"
]

robotics_responses = []

print("Surgical Robotics Responses:")
print("="*50)

for i, prompt in enumerate(robotics_test_prompts):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=config['generation_max_length'],
            temperature=config['generation_temperature'],
            top_p=config['generation_top_p'],
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    robotics_responses.append(response)
    print(f"\n{i+1}. Prompt: {prompt}")
    print(f"   Response: {response}")
    print("-" * 50)

## Calculate Diversity Metrics

In [ ]:
all_responses = dialog_responses + robotics_responses

print("\nDiversity Metrics:")
print("="*50)

# Overall metrics
distinct_1 = calculate_distinct_n(all_responses, 1)
distinct_2 = calculate_distinct_n(all_responses, 2)
avg_length = np.mean([len(text.split()) for text in all_responses])

print(f"\nOverall:")
print(f"  Distinct-1 (unigram diversity): {distinct_1:.4f}")
print(f"  Distinct-2 (bigram diversity): {distinct_2:.4f}")
print(f"  Avg Length: {avg_length:.2f} tokens")

# Dialog-specific
dialog_distinct_1 = calculate_distinct_n(dialog_responses, 1)
dialog_distinct_2 = calculate_distinct_n(dialog_responses, 2)
dialog_avg_length = np.mean([len(text.split()) for text in dialog_responses])

print(f"\nDialog:")
print(f"  Distinct-1: {dialog_distinct_1:.4f}")
print(f"  Distinct-2: {dialog_distinct_2:.4f}")
print(f"  Avg Length: {dialog_avg_length:.2f} tokens")

# Robotics-specific
robotics_distinct_1 = calculate_distinct_n(robotics_responses, 1)
robotics_distinct_2 = calculate_distinct_n(robotics_responses, 2)
robotics_avg_length = np.mean([len(text.split()) for text in robotics_responses])

print(f"\nRobotics:")
print(f"  Distinct-1: {robotics_distinct_1:.4f}")
print(f"  Distinct-2: {robotics_distinct_2:.4f}")
print(f"  Avg Length: {robotics_avg_length:.2f} tokens")

# Calculate token accuracy on validation sample
print("\nCalculating Token Accuracy on Validation Sample...")
sample_size = min(config['eval_sample_size'], len(val_loader.dataset))
correct_tokens = 0
total_tokens = 0

with torch.no_grad():
    for i, batch in enumerate(val_loader):
        if i * config['batch_size'] >= sample_size:
            break
        
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = outputs.logits.argmax(dim=-1)
        
        mask = labels != tokenizer.pad_token_id
        correct = (predictions == labels) & mask
        correct_tokens += correct.sum().item()
        total_tokens += mask.sum().item()

token_accuracy = correct_tokens / total_tokens if total_tokens > 0 else 0
print(f"  Token Accuracy: {token_accuracy:.4f} ({correct_tokens}/{total_tokens})")

## Save Evaluation Results

In [ ]:
results_path = os.path.join(config['output_dir'], "evaluation_results.json")

evaluation_results = {
    "overall_metrics": {
        "distinct_1": float(distinct_1),
        "distinct_2": float(distinct_2),
        "avg_response_length": float(avg_length),
        "token_accuracy": float(token_accuracy),
        "total_tokens_evaluated": int(total_tokens)
    },
    "dialog_metrics": {
        "distinct_1": float(dialog_distinct_1),
        "distinct_2": float(dialog_distinct_2),
        "avg_response_length": float(dialog_avg_length),
        "num_test_prompts": len(dialog_test_prompts)
    },
    "robotics_metrics": {
        "distinct_1": float(robotics_distinct_1),
        "distinct_2": float(robotics_distinct_2),
        "avg_response_length": float(robotics_avg_length),
        "num_test_prompts": len(robotics_test_prompts)
    },
    "test_examples": {
        "dialog_prompts": dialog_test_prompts,
        "dialog_responses": dialog_responses,
        "robotics_prompts": robotics_test_prompts,
        "robotics_responses": robotics_responses
    }
}

with open(results_path, 'w') as f:
    json.dump(evaluation_results, f, indent=2)

print(f"\n✓ Evaluation results saved to: {results_path}")

# Also save a human-readable text summary (as in original script)
summary_path = os.path.join(config['output_dir'], "evaluation_summary.txt")
with open(summary_path, 'w') as f:
    f.write("="*60 + "\n")
    f.write("GPT-2 CORE MODEL EVALUATION SUMMARY\n")
    f.write("="*60 + "\n\n")
    
    f.write("OVERALL METRICS\n")
    f.write("-"*60 + "\n")
    f.write(f"Distinct-1 (unigram diversity): {distinct_1:.4f}\n")
    f.write(f"Distinct-2 (bigram diversity): {distinct_2:.4f}\n")
    f.write(f"Average Response Length: {avg_length:.2f} tokens\n")
    f.write(f"Token Accuracy: {token_accuracy:.4f}\n\n")
    
    f.write("DIALOG METRICS\n")
    f.write("-"*60 + "\n")
    f.write(f"Distinct-1: {dialog_distinct_1:.4f}\n")
    f.write(f"Distinct-2: {dialog_distinct_2:.4f}\n")
    f.write(f"Average Length: {dialog_avg_length:.2f} tokens\n\n")
    
    f.write("SURGICAL ROBOTICS METRICS\n")
    f.write("-"*60 + "\n")
    f.write(f"Distinct-1: {robotics_distinct_1:.4f}\n")
    f.write(f"Distinct-2: {robotics_distinct_2:.4f}\n")
    f.write(f"Average Length: {robotics_avg_length:.2f} tokens\n\n")
    
    f.write("="*60 + "\n")
    f.write("SAMPLE RESPONSES\n")
    f.write("="*60 + "\n\n")
    
    f.write("DIALOG EXAMPLES:\n")
    f.write("-"*60 + "\n")
    for i, (prompt, response) in enumerate(zip(dialog_test_prompts, dialog_responses)):
        f.write(f"\n{i+1}. Prompt: {prompt}\n")
        f.write(f"   Response: {response}\n")
    
    f.write("\n" + "-"*60 + "\n")
    f.write("SURGICAL ROBOTICS EXAMPLES:\n")
    f.write("-"*60 + "\n")
    for i, (prompt, response) in enumerate(zip(robotics_test_prompts, robotics_responses)):
        f.write(f"\n{i+1}. Prompt: {prompt}\n")
        f.write(f"   Response: {response}\n")

print(f"✓ Human-readable summary saved to: {summary_path}")
print("\n✓ Training and Evaluation Complete!")

In [ ]:
# Save Results - Multiple Options
import shutil
import zipfile
from datetime import datetime

print("SAVING RESULTS")
print("="*50)

# Always save locally first
print(f"✓ Results saved locally at: {config['output_dir']}")

# Option 1: Try Google Drive if mounted
drive_mounted = os.path.exists('/content/drive/MyDrive')
if drive_mounted:
    try:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        drive_results_path = f"/content/drive/MyDrive/SAR_Training_Results_{timestamp}"
        
        print("Copying results to Google Drive...")
        shutil.copytree(config['output_dir'], drive_results_path)
        
        print(f"\n✓ Results saved to Google Drive!")
        print(f"Location: {drive_results_path}")
        
    except Exception as e:
        print(f"⚠️ Failed to copy to Google Drive: {e}")
        drive_mounted = False

# Option 2: Create downloadable ZIP file
print(f"\n📦 Creating downloadable ZIP file...")
zip_path = "/content/training_results.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add all files from output directory
    for root, dirs, files in os.walk(config['output_dir']):
        for file in files:
            file_path = os.path.join(root, file)
            arc_name = os.path.relpath(file_path, config['output_dir'])
            zipf.write(file_path, arc_name)

print(f"✓ ZIP file created: {zip_path}")

# Option 3: Download files directly
print(f"\n💾 DOWNLOAD OPTIONS:")
if drive_mounted:
    print(f"  • Google Drive: Files automatically saved to Drive")

print(f"  • ZIP Download: Right-click '{zip_path}' in files panel → Download")
print(f"  • Individual Files: Browse '{config['output_dir']}' in files panel")

print(f"\n📋 SAVED FILES SUMMARY:")
print(f"  • Best model: {config['best_model_path']}/")
print(f"  • Final model: {config['final_model_path']}/")
print(f"  • Training curves: {config['plot_path']}")
print(f"  • Evaluation results (JSON): {config['output_dir']}/evaluation_results.json")
print(f"  • Evaluation summary (TXT): {config['output_dir']}/evaluation_summary.txt")
print(f"  • Complete ZIP: {zip_path}")

print("\n✅ Training and Evaluation Complete!")
print("="*50)

## Save Results to Google Drive